# Layout-preserving PDF text extraction (PyMuPDF)

This notebook explains a small PDF text-extraction pipeline that tries to preserve the original visual layout (columns, spacing, and blank lines) when converting a PDF page to plain text.

The main idea:
1) Extract positioned text fragments (words or spans) with bounding boxes from each page.
2) Group fragments into “visual lines” based on Y-position.
3) Reconstruct each line into a fixed-width text line by mapping X-positions to character columns (spaces) or tab stops (tabs).
4) Insert blank lines when vertical gaps suggest there was whitespace between lines/blocks.

In [51]:
from __future__ import annotations
import re
import tempfile
from pathlib import Path
import fitz  # PyMuPDF
import statistics

## Global configuration constants

Paste the constants and regex definitions here.

Key parameters (adjust as needed):
- `PDF_LAYOUT_WIDTH_CHARS` — output width in characters (increase if columns collide).
- `PDF_LAYOUT_LINE_Y_TOL` — vertical tolerance in points for grouping words into lines (increase if single lines split).
- `PDF_LAYOUT_GAP_MULT` — gap multiplier that determines insertion of blank lines.
- `PDF_LAYOUT_USE_TABS` / `PDF_LAYOUT_TAB_SIZE_CHARS` — toggle/tab size for tab-based rendering.


In [52]:
PDF_LAYOUT_WIDTH_CHARS = 200     # increase if columns collide 
PDF_LAYOUT_LINE_Y_TOL = 4     # points; increase if a single line splits 
PDF_LAYOUT_GAP_MULT = 1.6        # bigger => fewer blank lines inserted

PDF_LAYOUT_USE_TABS = True         
PDF_LAYOUT_TAB_SIZE_CHARS = 8

---

## Helper: merge split numeric tokens

Purpose: join numeric tokens split during extraction (e.g., `'2'` + `'910,600'` → `'2910,600'`).

Notes:
- Uses regex `_NUM_HEAD_RE` and `_NUM_TAIL_RE`.
- Uses a height-scaled horizontal gap threshold to decide merging.



In [53]:
_NUM_HEAD_RE = re.compile(r"^\d{1,3}$")
_NUM_TAIL_RE = re.compile(r"^\d{3}[.,]\d+$")

In [54]:
def _merge_split_numbers_in_line(line, *, gap_factor: float = 2.0):
    """
    Joins patterns like: '2' + '910,600' => '2910,600'
    and '3' + '060,600' => '3060,600', when they are bery close on X.
    """
    if not line:
        return line

    merged = []
    i = 0
    while i < len(line):
        w1 = line[i]
        t1 = (w1[4] or "").strip()

        if i + 1 < len(line):
            w2 = line[i + 1]
            t2 = (w2[4] or "").strip()

            if _NUM_HEAD_RE.match(t1) and _NUM_TAIL_RE.match(t2):
                # Horizontal Gap between two words
                gap = float(w2[0]) - float(w1[2])
                h = max(1.0, float(w1[3]) - float(w1[1]))
                if 0 <= gap <= gap_factor * h:
                    new_text = t1 + t2  # removes the thousand separator (2 300,00 -> 2300,00)
                    new_item = (
                        float(w1[0]),
                        min(float(w1[1]), float(w2[1])),
                        float(w2[2]),
                        max(float(w1[3]), float(w2[3])),
                        new_text,
                        w1[5], w1[6], w1[7],
                    )
                    merged.append(new_item)
                    i += 2
                    continue

        merged.append(w1)
        i += 1

    return merged

## Helper: group words into visual lines

Purpose: convert a flat list of positioned words into line groups by comparing word Y-centers.

Notes:
- Sorts words by Y-center then X.
- Uses `y_tol` (vertical tolerance) to decide membership in the same line.
- Ensures words in a line are ordered left-to-right.

In [55]:
def _group_words_into_lines(words, y_tol: float):
    """
    words items from page.get_text("words"):
      (x0, y0, x1, y1, text, block_no, line_no, word_no)
    Groups words into visual lines by y-center.
    """
    # sort top-to-bottom then left-to-right
    words = sorted(words, key=lambda w: ((w[1] + w[3]) / 2.0, w[0]))

    lines = []
    cur = []
    cur_y = None

    for w in words:
        x0, y0, x1, y1, text = w[0], w[1], w[2], w[3], w[4]
        if not (text or "").strip():
            continue

        y_center = (y0 + y1) / 2.0
        if cur_y is None:
            cur = [w]
            cur_y = y_center
            continue

        if abs(y_center - cur_y) <= y_tol:
            cur.append(w)
        else:
            lines.append(sorted(cur, key=lambda ww: ww[0]))
            cur = [w]
            cur_y = y_center

    if cur:
        lines.append(sorted(cur, key=lambda ww: ww[0]))

    return lines


## Helper: render lines to fixed-width output

Purpose: map X positions to character columns or tab cells, insert blank lines when vertical gaps are large, and produce final text.

Behavior summary:
- Estimates typical line height via median of per-line heights.
- Computes global X range (`min_x`, `max_x`) for normalization.
- Inserts blank lines when `gap > gap_mult * line_height`.
- Option A — spaces: map X -> character column, pad with spaces. &rarr; Default
- Option B — tabs: map X -> tab cell index and join cells with `\t`. (opt-in; pass `use_tabs=True` explicitly)
- Calls `_merge_split_numbers_in_line` per line before rendering.

In [56]:
def _render_lines_to_fixed_width(
    lines,
    *,
    width_chars: int,
    gap_mult: float,
    use_tabs: bool = False,
    tab_size_chars: int = 8,
):
    """
    Layout-preserving text output.

    - use_tabs=False: Pads with spaces.
    - use_tabs=True: emits '\t' between tab-stops (coarse columns).
    """
    heights = []
    for line in lines:
        y0 = min(w[1] for w in line)
        y1 = max(w[3] for w in line)
        heights.append(max(1.0, y1 - y0))
    line_h = statistics.median(heights) if heights else 10.0

    all_x0 = [w[0] for ln in lines for w in ln]
    all_x1 = [w[2] for ln in lines for w in ln]
    min_x = min(all_x0) if all_x0 else 0.0
    max_x = max(all_x1) if all_x1 else 1.0
    content_w = max(1.0, max_x - min_x)

    out_lines = []
    prev_bottom = None

    # number of tab "cells" across the line when use_tabs=True
    tab_size_chars = max(1, int(tab_size_chars))
    n_tabs = max(2, (int(width_chars) + tab_size_chars - 1) // tab_size_chars)

    for ln in lines:
        top = min(w[1] for w in ln)
        bottom = max(w[3] for w in ln)
        ln = _merge_split_numbers_in_line(ln, gap_factor=2.0)
        if prev_bottom is not None:
            gap = top - prev_bottom
            if gap > gap_mult * line_h:
                blanks = max(1, int(round(gap / line_h)) - 1)
                out_lines.extend([""] * blanks)

        if not use_tabs:
            # -------- spaces to fixed columns --------
            s = ""
            for w in ln:
                x0, text = w[0], w[4]
                col = int(((x0 - min_x) / content_w) * (width_chars - 1))
                col = max(0, min(width_chars - 1, col))

                if len(s) < col:
                    s += " " * (col - len(s))
                else:
                    if s and not s.endswith(" "):
                        s += " "
                s += text
            out_lines.append(s.rstrip())
        else:
            # -------- tabs by tab-stops --------
            cells = [""] * n_tabs
            for w in ln:
                x0, text = w[0], w[4]

                # map x position -> tab cell index
                tcol = int(((x0 - min_x) / content_w) * (n_tabs - 1))
                tcol = max(0, min(n_tabs - 1, tcol))

                if cells[tcol]:
                    cells[tcol] += " " + text
                else:
                    cells[tcol] = text

            # trim trailing empty cells only (keep leading empties if any)
            last = -1
            for i in range(n_tabs - 1, -1, -1):
                if cells[i]:
                    last = i
                    break
            out_lines.append("\t".join(cells[: last + 1]) if last >= 0 else "")

        prev_bottom = bottom

    return "\n".join(out_lines).rstrip()


## Helper: extract spans filtered by minimum font size

Purpose: use `page.get_text("dict")` to extract spans and filter by `min_font_size` to ignore small artifacts.

Notes:
- Returns tuples shaped like `page.get_text("words")`.
- Useful to ignore footers, watermarks, or tiny labels.

In [57]:
def _items_from_dict_with_min_font(page, *, min_font_size: float):
    """
    Returns list of items shaped like 'words' tuples:
      (x0, y0, x1, y1, text, block_no, line_no, item_no)
    but built from spans filtered by font size.
    """
    d = page.get_text("dict")
    items = []
    item_no = 0

    for b_idx, block in enumerate(d.get("blocks", [])):
        if block.get("type") != 0:  # 0=text
            continue

        for l_idx, line in enumerate(block.get("lines", [])):
            for span in line.get("spans", []):
                size = float(span.get("size") or 0.0)
                if size < min_font_size:
                    continue

                text = (span.get("text") or "").strip()
                if not text:
                    continue

                x0, y0, x1, y1 = span["bbox"]
                items.append((x0, y0, x1, y1, text, b_idx, l_idx, item_no))
                item_no += 1

    return items

---

## Main function: layout-preserving extraction per page

Purpose: entry point `pdf_text_layout_preserving(pdf_path, max_pages, *, min_font_size=None)`.

Flow:
1. Open PDF with `fitz.open`.
2. For each page up to `max_pages`:
   - Extract items via `page.get_text("words")` or `_items_from_dict_with_min_font`.
   - Group into lines with `_group_words_into_lines`.
   - Render text with `_render_lines_to_fixed_width`.
   - Append a page header: `--- PAGE <n> (layout or layout>=fontX) ---`.
3. Close the document in a `finally` block.
4. Return combined string (or empty string on failure).


In [58]:
def pdf_text_layout_preserving(pdf_path: Path, max_pages: int, *, min_font_size: float | None = None) -> str:
    try:
        doc = fitz.open(str(pdf_path))
    except Exception:
        return ""

    parts = []
    try:
        n = min(len(doc), max_pages)
        for i in range(n):
            page = doc.load_page(i)

            if min_font_size is None:
                items = page.get_text("words") or []
            else:
                items = _items_from_dict_with_min_font(page, min_font_size=min_font_size)

            if not items:
                continue

            lines = _group_words_into_lines(items, y_tol=PDF_LAYOUT_LINE_Y_TOL)
            if not lines:
                continue

            text = _render_lines_to_fixed_width(
                lines,
                width_chars=PDF_LAYOUT_WIDTH_CHARS,
                gap_mult=PDF_LAYOUT_GAP_MULT,
                use_tabs=PDF_LAYOUT_USE_TABS,               
                tab_size_chars=PDF_LAYOUT_TAB_SIZE_CHARS,   
            )

            if text.strip():
                tag = f"layout>=font{min_font_size:g}" if min_font_size is not None else "layout"
                parts.append(f"\n--- PAGE {i+1} ({tag}) ---\n{text}")
    finally:
        try:
            doc.close()
        except Exception:
            pass

    return "\n".join(parts).strip()

---

---

## Page-level deduplication: header / footer removal

PDFs often repeat the same block of text on every page — a company name at the top, a page-number line at the bottom, a watermark phrase in the middle. When you concatenate pages for downstream processing (e.g. LLM context), those repetitions inflate the token count and pollute field extraction.

This section contains the helpers that detect and strip repeated blocks before the pages are joined.

Pipeline:
1. **Pre-clean each page** — collapse consecutive duplicate lines inside a single page.
2. **Trim repeated top block inside page 0** — catches OCR artefacts where the header is emitted twice on the first page.
3. **Compare pages 1…N against page 0 (base)** — remove the longest matching word-prefix (header), word-suffix (footer), and any long matching interior substring.
4. **Return** cleaned page bodies + the first removed top/bottom strings (useful as metadata).


### Text utility helpers

Three small normalisation helpers used throughout the dedup logic:
- `_normalize_whitespace` — collapses any run of whitespace to a single space and strips.
- `_lower_strip` — lowercase + normalize whitespace; used for case-insensitive comparisons.
- `_words` — tokenises a string into word/punctuation tokens via regex (same tokeniser used for prefix/suffix matching).


In [59]:
import re
from typing import List, Tuple

_word_re = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def _normalize_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def _lower_strip(s: str) -> str:
    return _normalize_whitespace(s).lower()

def _words(s: str) -> List[str]:
    return [m.group(0) for m in _word_re.finditer(s)]


### Helper: remove common word-prefix (repeated header)

Purpose: strip the longest common word-sequence that appears at the **start** of both `base` (page 0) and `text` (page N).

Notes:
- Comparison is case-insensitive and whitespace-normalised.
- The cut position is resolved against the *original* (un-normalised) `text` so the returned `remainder` preserves original spacing.
- Returns `(removed_prefix, remainder_text)` — `removed_prefix` is the raw slice that was cut.


In [60]:
def _remove_common_word_prefix(base: str, text: str) -> Tuple[str, str]:
    """
    Removes the longest common word-prefix between base and text.
    Returns (removed_prefix, remainder_text). `removed_prefix` preserves original text slice.
    """
    removed = ""
    if not base or not text:
        return removed, text

    base_tokens = _words(_lower_strip(base))
    text_tokens = _words(_lower_strip(text))
    if not base_tokens or not text_tokens:
        return removed, text

    i = 0
    max_i = min(len(base_tokens), len(text_tokens))
    while i < max_i and base_tokens[i] == text_tokens[i]:
        i += 1
    if i == 0:
        return removed, text

    orig_tokens = list(_word_re.finditer(text))
    if len(orig_tokens) < i:
        return removed, text
    cut_pos = orig_tokens[i - 1].end()
    removed = text[:cut_pos].rstrip()
    remainder = text[cut_pos:].lstrip()
    return removed, remainder


### Helper: remove common word-suffix (repeated footer)

Purpose: same as the prefix helper but operates from the **end** of the string — catches repeated footers (page numbers, legal disclaimers, etc.).

Notes:
- Compares token lists from the right.
- Cut position is the start of the first token in the matched tail.
- Returns `(removed_suffix, remainder_text)`.


In [61]:
def _remove_common_word_suffix(base: str, text: str) -> Tuple[str, str]:
    """
    Removes the longest common word-suffix between base and text.
    Returns (removed_suffix, remainder_text). `removed_suffix` preserves original text slice.
    """
    removed = ""
    if not base or not text:
        return removed, text

    base_tokens = _words(_lower_strip(base))
    text_tokens = _words(_lower_strip(text))
    if not base_tokens or not text_tokens:
        return removed, text

    i = 0
    max_i = min(len(base_tokens), len(text_tokens))
    # compare from the end
    while i < max_i and base_tokens[-1 - i] == text_tokens[-1 - i]:
        i += 1
    if i == 0:
        return removed, text

    orig_tokens = list(_word_re.finditer(text))
    if len(orig_tokens) < i:
        return removed, text
    start_pos = orig_tokens[-i].start()
    removed = text[start_pos:].rstrip()
    remainder = text[:start_pos].rstrip()
    return removed, remainder


### Helper: remove long matching interior substring (watermarks / mid-page repeats)

Purpose: when the prefix and suffix helpers find nothing, try to find a long contiguous fragment of `base` anywhere inside `text` and remove the first occurrence.

Notes:
- Only acts if `len(base_norm) >= min_chars` (default 20) to avoid false positives on short strings.
- First tries an exact (but case-insensitive) match on the normalised text.
- Falls back to a sliding word-window search, reducing the window until a match of at least `min_chars` characters is found.
- Returns `(removed_substring, remainder_text)`.


In [62]:
def _remove_case_insensitive_substring(base: str, text: str, min_chars: int = 20) -> Tuple[str, str]:
    """
    If a reasonably long normalized substring of `base` appears in `text`, remove the first occurrence.
    Returns (removed_substring, remainder_text). `removed_substring` preserves original text slice.
    """
    if not base or not text:
        return "", text

    base_norm = _lower_strip(base)
    if len(base_norm) < min_chars:
        return "", text

    text_norm = _lower_strip(text)
    if base_norm in text_norm:
        m = re.search(re.escape(base_norm), text, flags=re.IGNORECASE)
        if m:
            removed = text[m.start():m.end()].strip()
            remainder = (text[:m.start()] + text[m.end():]).strip()
            return removed, remainder

    # fallback: sliding word-window, shrinking until min_chars
    base_words = _words(base_norm)
    for window in range(len(base_words), 0, -1):
        candidate = " ".join(base_words[:window])
        if len(candidate) < min_chars:
            continue
        if candidate in text_norm:
            pattern = re.compile(re.escape(candidate), flags=re.IGNORECASE)
            m = pattern.search(text)
            if m:
                removed = text[m.start():m.end()].strip()
                remainder = (text[:m.start()] + text[m.end():]).strip()
                return removed, remainder
    return "", text


### Helper: collapse consecutive duplicate lines

Purpose: remove immediately repeated lines within a single page (a common PyMuPDF artefact where a text span is emitted twice).

Notes:
- Keeps the **first** occurrence; drops any line identical to the one immediately before it.
- Also collapses runs of blank lines to a single blank and strips leading/trailing blank lines.
- Applied as a pre-cleaning step on every page before cross-page dedup.


In [63]:
def _collapse_consecutive_duplicate_lines(text: str) -> str:
    """
    Remove immediate consecutive duplicate lines (keeps first occurrence).
    Also strip trailing/leading blank lines and collapse multi-blank to single blank.
    """
    if not text:
        return ""
    lines = text.splitlines()
    out_lines = []
    prev = None
    for ln in lines:
        if prev is None or ln != prev:
            out_lines.append(ln)
        prev = ln
    cleaned = "\n".join(out_lines)
    cleaned = re.sub(r"\n\s*\n+", "\n\n", cleaned).strip()
    return cleaned


### Helper: trim repeated block at the top of a page

Purpose: detect and remove a **second occurrence** of the opening block of text within the same page. Targets a specific OCR/export artefact where the page header is fused into the body and then repeated.

Notes:
- Only scans the first `max_scan_chars` characters of the page (default 2 000) as the candidate header.
- Searches for a second match of decreasing fragment sizes (down to `min_fragment_chars = 40`).
- A "second occurrence" qualifies only if the gap between end of first match and start of second is ≤ 200 characters (contiguous-ish).
- Applied only to page 0 (the base page) after `_collapse_consecutive_duplicate_lines`.


In [64]:
def _trim_repeated_block_at_top(text: str, max_scan_chars: int = 2000) -> str:
    """
    Heuristic: if the opening block of a page repeats immediately later in the same page,
    remove the second contiguous repetition. Useful when extraction yields the header twice.
    Only considers matches within max_scan_chars of the start for performance/safety.
    """
    if not text:
        return ""
    head = text[:max_scan_chars]
    head_norm = _lower_strip(head)
    min_fragment_chars = 40
    for size in range(len(head_norm), min_fragment_chars - 1, -1):
        frag = head_norm[:size]
        if len(frag) < min_fragment_chars:
            continue
        m = re.search(re.escape(frag), text, flags=re.IGNORECASE)
        if not m:
            continue
        first_end = m.end()
        m2 = re.search(re.escape(frag), text[first_end:], flags=re.IGNORECASE)
        if m2:
            start2 = first_end + m2.start()
            end2 = first_end + m2.end()
            gap = text[first_end:start2]
            if len(gap) <= 200:  # heuristic: small gap => repeated block
                return (text[:start2] + text[end2:]).strip()
    return text


### Main dedup function: `dedupe_pages_extract_tops_bottoms`

Purpose: orchestrates the full header/footer removal pipeline across a list of page texts.

Flow:
1. Pre-clean every page with `_collapse_consecutive_duplicate_lines`.
2. Run `_trim_repeated_block_at_top` on page 0 only.
3. Use the cleaned page 0 as `base` for all comparisons.
4. For each subsequent page:
   - Try `_remove_common_word_prefix` → removes header.
   - Try `_remove_common_word_suffix` → removes footer.
   - If neither matched, try `_remove_case_insensitive_substring` → removes mid-page watermark; classify it as top or bottom based on its position in the normalised text.
   - If only a prefix was removed, try suffix removal again on the remainder (catches pages with both).
   - Final collapse of duplicate lines on the cleaned middle.
5. Record the first non-empty top/bottom removed (returned as metadata for callers).

Returns `(core_pages, first_top_removed, first_bottom_removed)`.


In [65]:
def dedupe_pages_extract_tops_bottoms(
    page_texts: List[str],
    min_common_substring_chars: int = 40
) -> Tuple[List[str], str, str]:
    """
    Compare pages against page_texts[0] (base). Remove matching top (prefix),
    bottom (suffix) and long substring matches from subsequent pages.

    Returns (core_pages, first_top_removed, first_bottom_removed)
    - core_pages: list of cleaned page bodies (same length as page_texts)
    - first_top_removed: the first non-empty top (prefix) removed from any page
    - first_bottom_removed: the first non-empty bottom (suffix) removed from any page
    """
    if not page_texts:
        return [], "", ""

    # Step 1: pre-clean every page
    precleaned = [_collapse_consecutive_duplicate_lines((p or "").strip()) for p in page_texts]

    # Step 2: trim repeated header block inside page 0
    precleaned[0] = _trim_repeated_block_at_top(precleaned[0], max_scan_chars=3000)

    base = precleaned[0] or ""
    cores: List[str] = [base]  # page 0 kept as-is (already cleaned)
    first_top_removed: str = ""
    first_bottom_removed: str = ""

    for i in range(1, len(precleaned)):
        t_orig = precleaned[i]

        # Step 4a: strip header
        top_removed, middle = _remove_common_word_prefix(base, t_orig)

        # Step 4b: strip footer
        bottom_removed = ""
        bottom_candidate, middle_after_suffix = _remove_common_word_suffix(base, middle)
        if bottom_candidate:
            bottom_removed = bottom_candidate
            middle = middle_after_suffix

        # Step 4c: fallback — interior substring (watermark, mid-page repeat)
        if not top_removed and not bottom_removed:
            sub_removed, middle_after_sub = _remove_case_insensitive_substring(
                base, middle, min_chars=min_common_substring_chars
            )
            if sub_removed:
                norm_middle = _lower_strip(middle)
                norm_removed = _lower_strip(sub_removed)
                if norm_middle.startswith(norm_removed):
                    top_removed = sub_removed
                elif norm_middle.endswith(norm_removed):
                    bottom_removed = sub_removed
                else:
                    top_removed = sub_removed  # classify as top when position is ambiguous
                middle = middle_after_sub

        # Step 4d: second suffix pass if only prefix was removed
        if top_removed and not bottom_removed:
            bottom_candidate2, middle_after_suffix2 = _remove_common_word_suffix(base, middle)
            if bottom_candidate2:
                bottom_removed = bottom_candidate2
                middle = middle_after_suffix2

        # Record first seen top/bottom
        if top_removed and not first_top_removed:
            first_top_removed = top_removed.strip()
        if bottom_removed and not first_bottom_removed:
            first_bottom_removed = bottom_removed.strip()

        middle = _collapse_consecutive_duplicate_lines(middle)
        cores.append((middle or "").strip())

    return cores, first_top_removed, first_bottom_removed


---

## Higher-level extraction functions

The functions below sit above the layout engine. They add fallback strategies, quality gating, and a per-page list API used by `build_attachment_evidence`.


### Helper: `is_text_good`

Purpose: lightweight quality gate — returns `True` if the extracted text looks like real readable content rather than binary garbage or an empty extraction.

Checks:
- Minimum total character count (`min_chars`, default 800).
- Minimum ratio of alphabetic characters to total characters (`min_alpha_ratio`, default 0.15).
- Maximum ratio of replacement characters (`\ufffd`) and low ASCII control bytes — above 1 % signals garbled extraction.


In [66]:
def is_text_good(text: str, min_chars: int = 800, min_alpha_ratio: float = 0.15) -> bool:
    if not text:
        return False
    if len(text) < min_chars:
        return False

    letters = sum(ch.isalpha() for ch in text)
    ratio = letters / max(len(text), 1)

    bad = text.count("\ufffd") + len(re.findall(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", text))
    bad_ratio = bad / max(len(text), 1)

    return ratio >= min_alpha_ratio and bad_ratio < 0.01


### `pdf_text_best_effort`

Purpose: wraps `pdf_text_layout_preserving` with a fallback to `pypdf`'s plain-text extractor for cases where PyMuPDF returns nothing (encrypted PDFs, some scans with embedded text layers).

Flow:
1. Try `pdf_text_layout_preserving`; return immediately if non-empty.
2. If `min_font_size` was set, give up (font-filtered extraction failing means there is genuinely nothing above that threshold).
3. Otherwise fall back to `PdfReader.extract_text()` page-by-page.


In [67]:
from pypdf import PdfReader

def pdf_text_best_effort(pdf_path: Path, max_pages: int, *, min_font_size: float | None = None) -> str:
    text = pdf_text_layout_preserving(pdf_path, max_pages=max_pages, min_font_size=min_font_size)
    if text.strip():
        return text

    # Font-size filter was set: no point falling back — nothing above threshold.
    if min_font_size is not None:
        return ""

    try:
        reader = PdfReader(str(pdf_path))
        parts = []
        for i, page in enumerate(reader.pages[:max_pages], start=1):
            t = page.extract_text() or ""
            if t.strip():
                parts.append(f"\n--- PAGE {i} (text) ---\n{t}")
        return "\n".join(parts).strip()
    except Exception:
        return ""


### `pdf_text_first_pages_best_effort`

Purpose: top-level convenience wrapper used by `extract_text_best_effort` for PDFs. Adds a third-tier fallback on top of `pdf_text_best_effort`.

Flow:
1. Call `pdf_text_best_effort` (layout → pypdf chain).
2. If the result fails `is_text_good` (< 200 chars or < 10 % alpha), try PyMuPDF's plain `get_text("text")` mode as a last resort — this mode skips the bounding-box pipeline entirely and can succeed on some PDFs where the word-extraction API returns nothing.


In [68]:
def pdf_text_first_pages_best_effort(pdf_path: Path, max_pages: int = 10, min_font_size: int = 10) -> str:
    text = pdf_text_best_effort(pdf_path, max_pages=max_pages, min_font_size=min_font_size)

    # Third-tier fallback: PyMuPDF plain text mode
    if not is_text_good(text, min_chars=200, min_alpha_ratio=0.10):
        try:
            doc = fitz.open(str(pdf_path))
            parts = []
            for i in range(min(len(doc), max_pages)):
                t = doc.load_page(i).get_text("text") or ""
                if t.strip():
                    parts.append(f"\n--- PAGE {i + 1} (text) ---\n{t}")
            if parts:
                text = "\n".join(parts)
        except Exception:
            pass

    return text or ""


### `pdf_text_per_page`

Purpose: returns a **list** of per-page text strings (rather than a single concatenated string). Used by `build_attachment_evidence` so the caller can run `dedupe_pages_extract_tops_bottoms` on the raw list before joining.

Flow:
1. Preferred: open with PyMuPDF, extract words per page, run through `_group_words_into_lines` + `_render_lines_to_fixed_width`. Empty pages become empty strings.
2. Fallback: `pypdf` `extract_text()` per page — returns whatever pypdf can extract.
3. Returns `[]` only if both methods raise.


In [69]:
def pdf_text_per_page(pdf_path: Path, max_pages: int) -> list[str]:
    """
    Returns per-page text in layout-preserving form (preferred) with fallback.
    """
    # Preferred: layout-preserving per page
    try:
        doc = fitz.open(str(pdf_path))
        out = []
        for i in range(min(len(doc), max_pages)):
            page = doc.load_page(i)
            words = page.get_text("words") or []
            if words:
                lines = _group_words_into_lines(words, y_tol=PDF_LAYOUT_LINE_Y_TOL)
                out.append(
                    _render_lines_to_fixed_width(
                        lines,
                        width_chars=PDF_LAYOUT_WIDTH_CHARS,
                        gap_mult=PDF_LAYOUT_GAP_MULT,
                        use_tabs=PDF_LAYOUT_USE_TABS,
                        tab_size_chars=PDF_LAYOUT_TAB_SIZE_CHARS,
                    )
                )
            else:
                out.append("")
        return out
    except Exception:
        pass

    # Fallback: pypdf per page
    try:
        reader = PdfReader(str(pdf_path))
        return [(p.extract_text() or "") for p in reader.pages[:max_pages]]
    except Exception:
        return []


---

## Entry point: `build_attachment_evidence` (PDF path)

Purpose: the single function callers invoke per file. For a PDF it orchestrates the full pipeline — text extraction, per-page quality scoring, header/footer dedup, and page rasterisation — and returns a uniform evidence dict.

This section covers only the **PDF branch** (`p.suffix.lower() == ".pdf"`). Other file types (DOCX, XLSX, CSV) follow separate branches not documented here.


### Helper: `count_good_chars`

Purpose: counts alphanumeric characters in a string. Used as a fast proxy for extraction quality — a page with very few alphanumeric chars is likely blank or garbled.

Note: intentionally simpler than `is_text_good` (no ratio, no bad-char check) because here we want a raw count to compare against `min_good_chars_per_page`.


In [70]:
def count_good_chars(text: str) -> int:
    return sum(ch.isalnum() for ch in (text or ""))


### Helper: `doc_type_label`

Purpose: maps a file extension to a short uppercase label stored in the evidence dict. Purely cosmetic — used by downstream logging and UI display.


In [71]:
def doc_type_label(p: Path) -> str:
    ext = p.suffix.lower()
    if ext in (".pdf", ".PDF"):
        return "PDF"
    if ext == ".xlsx":
        return "XLSX"
    if ext == ".csv":
        return "CSV"
    if ext == ".docx":
        return "DOCX"
    if ext == ".doc":
        return "DOC"
    if ext == ".xls":
        return "XLS"
    return ext.upper().lstrip(".")


### Helper: `pdf_pages_to_image_data_urls`

Purpose: rasterises each PDF page to a PNG and returns it as a base64 `data:image/png` URL. These are passed to the vision model when `allow_text_only` is `False`.

Notes:
- Uses PyMuPDF's `page.get_pixmap(dpi=dpi)` — 100 dpi is enough for LLM vision, higher costs more tokens.
- Each URL is self-contained (no external reference needed).
- Called via `file_to_image_data_urls`, which routes by extension; for PDFs it delegates directly here.


In [72]:
import base64

def pdf_pages_to_image_data_urls(pdf_path: Path, max_pages: int, dpi: int = 100) -> list[str]:
    doc = fitz.open(str(pdf_path))
    urls: list[str] = []
    for i in range(min(len(doc), max_pages)):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=dpi)
        b64 = base64.b64encode(pix.tobytes("png")).decode("utf-8")
        urls.append(f"data:image/png;base64,{b64}")
    return urls


### `build_attachment_evidence` — PDF branch

Flow for a `.pdf` input:
1. **Split into distinct invoices** — some source PDFs bundle more than one
   invoice, and each invoice page can be duplicated as an "Original" and a
   "Cópia" (copy). Pages flagged as a copy are discarded; remaining
   "Original" pages are grouped by invoice number (e.g. `231/4302108`) into
   separate single-invoice PDFs via `_split_original_invoices`.
2. For **each** resulting single-invoice PDF:
   1. **`pdf_text_per_page`** — extract text as a list of per-page strings
      (layout-preserving, with pypdf fallback).
   2. **`count_good_chars` per page** — score each page; take `max_good` as
      the quality signal.
   3. **`dedupe_pages_extract_tops_bottoms`** — strip repeated headers/footers;
      returns `(clean_pages, top_page, bot_page)`.
   4. **Reassemble** — join cleaned page bodies, then sandwich the recovered
      header (`top_page`) and footer (`bot_page`) back around the body.
      This preserves metadata (e.g. company name, document title) while
      removing per-page repetition.
   5. **`allow_text_only`** — `True` if `max_good >= min_good_chars_per_page`;
      signals to the caller that vision is not needed.
   6. **`pdf_pages_to_image_data_urls`** — rasterise pages regardless (caller
      decides whether to use them based on `allow_text_only`).

Returns a **list** of evidence dicts (one per distinct invoice found), each
with keys: `path`, `filename`, `doc_type`, `extracted_text`, `allow_text_only`,
`good_score`, `image_urls`. A PDF containing a single invoice returns a
one-item list.

In [73]:
from typing import Optional

_INVOICE_NUMBER_RE = re.compile(r"\b\d{2,4}/\d{5,8}\b")


def _page_is_copy(page_text: str) -> bool:
    """True if the page body flags itself as a 'Cópia' (copy)."""
    return "cópia" in page_text.lower() or "copia" in page_text.lower()


def _page_invoice_number(page_text: str) -> str | None:
    """Extracts the invoice number (e.g. '231/4302108') from a page body."""
    m = _INVOICE_NUMBER_RE.search(page_text)
    return m.group(0) if m else None


def _split_original_invoices(pdf_path: Path, max_pages: int) -> list[Path]:
    """
    Splits a multi-invoice PDF into one single-invoice PDF per distinct
    invoice number, keeping only 'Original' pages (discarding 'Cópia' pages).

    Returns a list of paths to temporary single-invoice PDFs, one per
    distinct invoice number found (in order of first appearance).
    """
    doc = fitz.open(str(pdf_path))
    try:
        n = min(len(doc), max_pages)

        groups: dict[str, list[int]] = {}
        order: list[str] = []
        for i in range(n):
            page_text = doc.load_page(i).get_text("text") or ""
            if _page_is_copy(page_text):
                continue

            invoice_no = _page_invoice_number(page_text) or f"page-{i}"
            if invoice_no not in groups:
                groups[invoice_no] = []
                order.append(invoice_no)
            groups[invoice_no].append(i)

        out_paths: list[Path] = []
        tmp_dir = Path(tempfile.mkdtemp(prefix="invoice_split_"))
        for invoice_no in order:
            sub_doc = fitz.open()
            for page_no in groups[invoice_no]:
                sub_doc.insert_pdf(doc, from_page=page_no, to_page=page_no)
            safe_name = re.sub(r"[^A-Za-z0-9_-]", "_", invoice_no)
            out_path = tmp_dir / f"{pdf_path.stem}__{safe_name}.pdf"
            sub_doc.save(str(out_path))
            sub_doc.close()
            out_paths.append(out_path)

        return out_paths
    finally:
        doc.close()


def _build_single_pdf_evidence(
    p: Path,
    *,
    max_pages_text: int,
    max_pages_vision: int,
    dpi: int,
    min_good_chars_per_page: int,
) -> dict:
    """Builds one evidence dict for a single-invoice PDF (no splitting)."""
    # Step 1: extract text as a list — one string per page
    page_texts = pdf_text_per_page(p, max_pages=max_pages_text)

    # Step 2: score each page by alphanumeric char count
    good_counts = [count_good_chars(t) for t in page_texts]
    max_good = max(good_counts, default=0)

    # Step 3: strip repeated headers / footers across pages
    clean_pages, top_page, bot_page = dedupe_pages_extract_tops_bottoms(
        page_texts, min_common_substring_chars=100
    )

    # Step 4: join cleaned bodies
    extracted_body = "\n".join((t or "").strip() for t in clean_pages).strip()

    # Step 4 (cont.): sandwich header + body + footer
    parts = []
    if top_page:
        parts.append(top_page.strip())
    if extracted_body:
        parts.append(extracted_body)
    if bot_page:
        parts.append(bot_page.strip())
    extracted_text = "\n\n".join(parts).strip()

    # Step 5: quality gate
    allow_text_only = max_good >= min_good_chars_per_page
    good_score = max_good

    urls: list[str] = []
    if not allow_text_only:
        # Step 6: rasterise pages for vision
        urls = pdf_pages_to_image_data_urls(p, max_pages=max_pages_vision, dpi=dpi)

    return {
        "path": str(p),
        "filename": p.name,
        "doc_type": doc_type_label(p),
        "extracted_text": extracted_text or "",
        "allow_text_only": bool(allow_text_only and (extracted_text or "").strip()),
        "good_score": int(good_score),
        "image_urls": urls or [],
    }


def build_attachment_evidence(
    p: Path,
    *,
    max_pages_text: int,
    max_pages_vision: int,
    dpi: int,
    min_good_chars_per_page: int,
    min_font_size: int,
) -> list[dict]:
    if p.suffix.lower() != ".pdf":
        # Non-PDF branches (DOCX, XLSX, CSV) — out of scope for this notebook
        print(f"Non-PDF branch not covered in this notebook: {p.suffix}")
        return []

    # Split multi-invoice PDFs into one single-invoice PDF per distinct
    # invoice number, discarding 'Cópia' pages.
    invoice_pdf_paths = _split_original_invoices(p, max_pages=max_pages_text)

    return [
        _build_single_pdf_evidence(
            invoice_pdf_path,
            max_pages_text=max_pages_text,
            max_pages_vision=max_pages_vision,
            dpi=dpi,
            min_good_chars_per_page=min_good_chars_per_page,
        )
        for invoice_pdf_path in invoice_pdf_paths
    ]

---

## Example usage 

In [74]:
from pathlib import Path

#attachment_path = Path("docs/FT-4302108.pdf")
attachment_path = Path("docs/two_invoices_one_copy.pdf")
print(attachment_path)

# Returns one evidence dict per distinct invoice found in the PDF
# ('Cópia' pages are discarded automatically).
evidences = build_attachment_evidence(
    attachment_path,
    max_pages_text=10,
    max_pages_vision=5,
    dpi=100,
    min_good_chars_per_page=300,
    min_font_size=10,
)
print(f"Found {len(evidences)} distinct invoice(s)")

for evidence in evidences:
    print("\nfilename  :", evidence["filename"])
    print("doc_type  :", evidence["doc_type"])
    print("good_score:", evidence["good_score"])
    print("allow_text:", evidence["allow_text_only"])
    print("images    :", len(evidence["image_urls"]), "page(s)")
    print("--- extracted text (first 1000 chars) ---")
    print(evidence["extracted_text"][:1000])

docs\two_invoices_one_copy.pdf
Found 2 distinct invoice(s)

filename  : two_invoices_one_copy__231_4302108.pdf
doc_type  : PDF
good_score: 1855
allow_text: True
images    : 0 page(s)
--- extracted text (first 1000 chars) ---
Factura		231/4302108
																Data	de	Emissão:			03-02-2023
ACTIVIDADES				TRANSITÁRIAS,				LDA.
																						Original
RETAIL	PARK	- Rua	Azores		Park,	104
Armazéns	1.9	e 1.10	9500-702		PONTA	DELGADA					Exmo(s)		Senhor(es)		:
Importação:		rui@ilhazores.com			.	Tel 296	683	190
Exportação:		mario@ilhazores.com				. Tel	296 683	156
											AZORFISK			UNIPESSOAL,				LDA
		CONTRIBUINTE			N.º 512	046	158
Sociedade	por	Quotas	Capital	Social 100.000,00		EUROS
Matric. Conserv.		do Registo	Comercial		de Ponta	Delgada	sob	o n.º	1977
											CANADA		DA	LAPINHA		S/N
											RABO	DE	PEIXE
											9600		RIBEIRA		GRANDE

													Número		de	Contribuinte:					513791345

Processo:			4322077		Condição			Pagamento:				30	Dias			Data	Vencimento:			05-03-

STRUCTURED OUTPUTS


After having the pdf parsed to text (can still work on the parser to tailor it for invoices, but good first version): we come to structured outputs

In [75]:
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from llm_maker import make_llm

load_dotenv()
llm = make_llm(model="gpt-4o-mini", temperature=0.7)


In [76]:
from pydantic import BaseModel, Field, field_validator
from datetime import datetime


class InvoiceLine(BaseModel):
    """A single line item (row) in an invoice."""
    description: str = Field(description="Description of the product or service")
    quantity: float = Field(description="Quantity billed")
    units: str = Field(description="Unit of measure (e.g. UN, Kgs)")
    price: float = Field(description="Unit price")
    value: float = Field(description="Total value for the line (quantity * price)")
    vat: float = Field(description="VAT / IVA percentage or amount applied to the line")


class Invoice(BaseModel):
    """Information about a person."""
    client_name: str = Field(description="The name of the client")
    date: str = Field(description="Day the invoice was issued, formatted as DD-MM-YYYY")
    price: str = Field(description="Total cost on the invoice")
    invoice_rows: list[InvoiceLine] = Field(description="Line items (rows) in the invoice")

    @field_validator("date")
    @classmethod
    def parse_date(cls, value: str) -> str:
        # Normalize/validate against the expected DD-MM-YYYY format; raises if malformed.
        datetime.strptime(value, "%d-%m-%Y")
        return value

In [77]:
# Create an LLM with structured output
structured_llm = llm.with_structured_output(Invoice)

# Extract structured data independently for each distinct invoice
results = [structured_llm.invoke(evidence["extracted_text"]) for evidence in evidences]

for evidence, result in zip(evidences, results):
    print(f"\n=== {evidence['filename']} ===")
    print("Structured output:", result)
    print("Client Name:", result.client_name)
    print("Date:", result.date)
    print("Price:", result.price)
    for i, row in enumerate(result.invoice_rows):
        print(f"  Invoice row {i}: {row}")


=== two_invoices_one_copy__231_4302108.pdf ===
Structured output: client_name='DUMAR/SCANFISK' date='03-02-2023' price='3757.60 €' invoice_rows=[InvoiceLine(description='FRETE MARITIMO EXPORTAÇÃO', quantity=1.0, units='UN', price=1975.0, value=1975.0, vat=0.0), InvoiceLine(description='CAMIONAGEM MARITIMA EXPORTAÇÃO', quantity=1.0, units='UN', price=100.0, value=100.0, vat=16.0), InvoiceLine(description='JAP DESPESAS MARITIMA EXPORTAÇÃO', quantity=1.0, units='UN', price=110.0, value=110.0, vat=0.0), InvoiceLine(description='TAXA LIGAÇÃO ENERGIA CAIS LISBOA', quantity=1.0, units='UN', price=85.0, value=85.0, vat=16.0), InvoiceLine(description='CAMIONAGEM MARITIMA IMPORTAÇÃO', quantity=1.0, units='UN', price=100.0, value=100.0, vat=16.0), InvoiceLine(description='DESCONSOLIDAÇÃO', quantity=1.0, units='UN', price=150.0, value=150.0, vat=16.0), InvoiceLine(description='ENTREGA DOMICILIO', quantity=1.0, units='UN', price=750.0, value=750.0, vat=16.0), InvoiceLine(description='SEGURO CARGA 